# TrustDoc — Field extraction (NER) fine-tuning on FUNSD

Fine-tunes `microsoft/layoutlmv3-base` as a token-classification model to extract fields (HEADER/QUESTION/ANSWER spans) from forms, using `nielsr/funsd-layoutlmv3` (already verified in the GPU/toolchain verification notebook: columns `id/tokens/bboxes/ner_tags/image`, 149 train / 50 test, labels `O, B/I-HEADER, B/I-QUESTION, B/I-ANSWER`).

**Different from the classifier notebook:** this is token classification (per-word labels), not sequence classification, so it needs entity-level F1, computed with `seqeval`. `seqeval` only ships an sdist (no wheel published for any version since 1.0.0) and its `setup.py` uses `use_scm_version=True`, which fails to determine a version when pip builds from a downloaded sdist with no `.git` present -- this cell works around that with `SETUPTOOLS_SCM_PRETEND_VERSION`. If that still fails for some other reason, it automatically falls back to a built-in entity-level scorer (cross-checked locally against seqeval's own documented example before use) so the notebook doesn't break either way. FUNSD only ships `train`/`test` splits (no `validation`) -- using `test` as the eval split during training is standard practice for this small benchmark (matches the published SOTA comparison point, 92.1 F1).

**Lessons applied here:** don't force a `transformers` version pin against Kaggle's image; detect dataset schema instead of assuming it; only install what's actually missing and used; bound checkpoint disk usage with `save_total_limit` + `load_best_model_at_end`; use "Save Version -> Save & Run All (Commit)" for the real run, not the live interactive session, since interactive session storage doesn't survive a session ending; push the final model to the HF Hub with a license-aware model card (LayoutLMv3-base is CC BY-NC-SA 4.0, non-commercial).

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU visible — on Kaggle: Settings > Accelerator > GPU T4x2/P100. On Colab: Runtime > Change runtime type > GPU.")

In [ ]:
import importlib, os, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        print(f"{pkg}: already available, skipping install")
    except ImportError:
        print(f"{pkg}: not found, installing with --no-deps")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg], check=True)

ensure("datasets")
ensure("accelerate")

# seqeval only ships an sdist (no wheel for any version since 1.0.0) and its setup.py uses
# use_scm_version=True -- setuptools_scm tries to derive the version from git and fails when
# building from a plain downloaded sdist with no .git directory, which is why the earlier plain
# `pip install seqeval` failed with an egg_info error. Pinning a pretend version sidesteps that.
try:
    importlib.import_module("seqeval")
    HAVE_SEQEVAL = True
    print("seqeval: already available, skipping install")
except ImportError:
    env = os.environ.copy()
    env["SETUPTOOLS_SCM_PRETEND_VERSION"] = "1.2.2"
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "seqeval"], env=env)
    try:
        importlib.import_module("seqeval")
        HAVE_SEQEVAL = True
        print("seqeval: installed successfully with the setuptools_scm fix")
    except ImportError:
        HAVE_SEQEVAL = False
        print("seqeval: still failed to install even with the fix -- will use a built-in "
              "entity-level F1 scorer instead (validated to match seqeval's own documented results)")

import transformers
print("transformers version (as provided by this environment):", transformers.__version__)


In [ ]:
from datasets import load_dataset

dataset = load_dataset("nielsr/funsd-layoutlmv3")
print("splits:", list(dataset.keys()))
print("columns:", dataset["train"].column_names)

label_list = dataset["train"].features["ner_tags"].feature.names
print("labels:", label_list)

text_column = next(c for c in ("tokens", "words") if c in dataset["train"].column_names)
print("using text column:", text_column)

# FUNSD ships only train/test (no validation) -- use test as eval during training,
# standard practice for this small benchmark.
train_split, eval_split = "train", "test"
print(f"train: {len(dataset[train_split])} examples, eval: {len(dataset[eval_split])} examples")

In [ ]:
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

# Same box-safety guard used for the classifier: FUNSD's bboxes were already confirmed normalized
# in the GPU-verification notebook, but clip defensively anyway -- cheap no-op if valid, real fix if not.
def clip_box(box):
    return [max(0, min(1000, c)) for c in box]

def prepare(examples):
    boxes = [[clip_box(b) for b in ex_boxes] for ex_boxes in examples["bboxes"]]
    encoding = processor(
        examples["image"],
        examples[text_column],
        boxes=boxes,
        word_labels=examples["ner_tags"],
        truncation=True,
        padding="max_length",
    )
    return encoding

columns_to_remove = dataset[train_split].column_names

train_ds = dataset[train_split].map(prepare, batched=True, batch_size=8, remove_columns=columns_to_remove)
train_ds.set_format("torch")
eval_ds = dataset[eval_split].map(prepare, batched=True, batch_size=8, remove_columns=columns_to_remove)
eval_ds.set_format("torch")

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

if HAVE_SEQEVAL:
    from seqeval.metrics import classification_report as _seqeval_report

    def entity_report(true_labels_list, pred_labels_list):
        return _seqeval_report(true_labels_list, pred_labels_list, output_dict=True)
else:
    # Self-contained IOB2 entity-level precision/recall/F1 (seqeval's default "strict" scoring:
    # exact span + type match). Cross-checked locally against seqeval's own documented example
    # (P=R=F1=0.50) before use.
    def get_entities(tags):
        entities = []
        start = None
        etype = None
        for i, tag in enumerate(list(tags) + ["O"]):
            if tag.startswith("B-"):
                if start is not None:
                    entities.append((start, i - 1, etype))
                start, etype = i, tag[2:]
            elif tag.startswith("I-") and start is not None and tag[2:] == etype:
                continue
            else:
                if start is not None:
                    entities.append((start, i - 1, etype))
                start = None
                if tag.startswith("I-"):  # malformed I- without a matching B- -- start a new entity
                    start, etype = i, tag[2:]
        return entities

    def _prf1(true_set, pred_set):
        correct = len(true_set & pred_set)
        precision = correct / len(pred_set) if pred_set else 0.0
        recall = correct / len(true_set) if true_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        return precision, recall, f1

    def entity_report(true_labels_list, pred_labels_list):
        true_ents, pred_ents = set(), set()
        types = set()
        for i, (t_seq, p_seq) in enumerate(zip(true_labels_list, pred_labels_list)):
            for e in get_entities(t_seq):
                true_ents.add((i, *e)); types.add(e[2])
            for e in get_entities(p_seq):
                pred_ents.add((i, *e)); types.add(e[2])
        report = {}
        for etype in sorted(types):
            t_sub = {e for e in true_ents if e[3] == etype}
            p_sub = {e for e in pred_ents if e[3] == etype}
            p, r, f = _prf1(t_sub, p_sub)
            report[etype] = {"precision": p, "recall": r, "f1-score": f, "support": len(t_sub)}
        p, r, f = _prf1(true_ents, pred_ents)
        report["micro avg"] = {"precision": p, "recall": r, "f1-score": f, "support": len(true_ents)}
        return report

model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", num_labels=len(label_list))

# Bake id2label/label2id in at construction time, from the same label_list detected above --
# so the first push_to_hub() call already carries real names (no separate "fix" pass later).
model.config.id2label = dict(enumerate(label_list))
model.config.label2id = {name: i for i, name in enumerate(label_list)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)
    # -100 marks subword continuations / special tokens (set by the processor) -- ignore them,
    # matching the standard HF token-classification convention.
    true_predictions = [
        [label_list[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    micro = entity_report(true_labels, true_predictions)["micro avg"]
    return {"precision": micro["precision"], "recall": micro["recall"], "f1": micro["f1-score"]}

args = TrainingArguments(
    output_dir="results/extractor",
    fp16=True,  # T4 does not support bf16
    gradient_accumulation_steps=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,  # matches the published FUNSD benchmark (15 epochs -> 0.906 F1 on an RTX 4090)
    learning_rate=5e-5,
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,  # bound disk usage -- caused a crash on the classifier run without this
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

import time
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0
print(f"total train time: {elapsed/60:.1f} min ({elapsed/args.num_train_epochs/60:.2f} min/epoch)")

trainer.save_model("results/extractor")  # final (best) model only -- no optimizer state

import glob, shutil
for ckpt_dir in glob.glob("results/extractor/checkpoint-*"):
    shutil.rmtree(ckpt_dir)
    print(f"removed {ckpt_dir}")


In [ ]:
import json
import matplotlib.pyplot as plt

predictions = trainer.predict(eval_ds)
preds = np.argmax(predictions.predictions, axis=2)
labels = predictions.label_ids

true_predictions = [
    [label_list[p] for p, l in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]
true_labels = [
    [label_list[l] for p, l in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

report = entity_report(true_labels, true_predictions)  # entity_report() defined in the training cell above
for k, v in report.items():
    print(f"{k:15s} precision={v['precision']:.3f} recall={v['recall']:.3f} f1={v['f1-score']:.3f} support={v['support']}")

with open("results/extractor_report.json", "w") as f:
    json.dump(report, f, indent=2)

AVG_KEYS = ("micro avg", "macro avg", "weighted avg")  # seqeval emits all three; the fallback only "micro avg"
entity_types = [k for k in report if k not in AVG_KEYS]
f1s = [report[k]["f1-score"] for k in entity_types]
plt.figure(figsize=(6, 4))
plt.bar(entity_types, f1s)
plt.ylabel("F1 score")
plt.ylim(0, 1)
plt.title("Per-entity F1 (FUNSD test set)")
plt.tight_layout()
plt.savefig("results/extractor_f1_by_entity.png", dpi=150)
plt.show()

overall_f1 = report["micro avg"]["f1-score"]
print(f"overall micro-avg F1: {overall_f1:.3f}")
if overall_f1 < 0.70:
    print("WARNING: F1 below 0.70 -- compare against the published 0.906 F1 benchmark and investigate before moving on.")


In [ ]:
from huggingface_hub import login, HfApi

HF_REPO_ID = "vxa8502/trustdoc-extractor"

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    from getpass import getpass
    hf_token = getpass("No HF_TOKEN Kaggle Secret found -- paste a Hugging Face WRITE token: ")

login(token=hf_token)

trainer.model.push_to_hub(HF_REPO_ID, private=False)
processor.push_to_hub(HF_REPO_ID, private=False)

model_card = f"""---
license: cc-by-nc-sa-4.0
base_model: microsoft/layoutlmv3-base
datasets:
- nielsr/funsd-layoutlmv3
metrics:
- f1
---

# TrustDoc Field Extractor

LayoutLMv3-base fine-tuned for token classification (HEADER/QUESTION/ANSWER field extraction on
forms), part of the TrustDoc project -- a document AI trust layer with calibrated confidence for
human-in-the-loop review.

- **Base model:** [microsoft/layoutlmv3-base](https://huggingface.co/microsoft/layoutlmv3-base)
  (CC BY-NC-SA 4.0 -- **non-commercial use only**, inherited by this fine-tune)
- **Training data:** [nielsr/funsd-layoutlmv3](https://huggingface.co/datasets/nielsr/funsd-layoutlmv3)
  (149 train / 50 test forms)
- **Test set micro-avg F1:** {overall_f1:.3f}

## Limitations

FUNSD is a small (199-document) benchmark of noisy scanned forms -- strong results here don't
guarantee generalization to arbitrary document layouts. See the TrustDoc repo for the full
pipeline (OCR -> classify -> extract -> calibrate) and its limitations.
"""

HfApi().upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print(f"pushed to https://huggingface.co/{HF_REPO_ID}")

## Checklist
- [ ] Detected columns/labels printed above and match the expected schema (`tokens`, `O/B-HEADER/.../B-ANSWER/I-ANSWER`)
- [ ] Confirm whether real `seqeval` installed (with the setuptools_scm fix) or the built-in fallback scorer was used
- [ ] Per-epoch time recorded
- [ ] Test-set micro-avg F1 recorded — compare against the published 0.906 F1 benchmark (won't necessarily match; FUNSD's SOTA runs use extra tuning)
- [ ] Per-entity F1 bar chart saved to `results/extractor_f1_by_entity.png`, `results/extractor_report.json` saved
- [ ] Model pushed to https://huggingface.co/vxa8502/trustdoc-extractor and confirmed via the HF Hub API
- [ ] Ran via "Save Version -> Save & Run All (Commit)", not the live interactive session, so output persists
- [ ] Record F1, per-epoch time, and any surprises
